In [1]:
!pip install google-cloud-bigquery google-cloud-bigquery-storage pandas db-dtypes

In [3]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
from datetime import datetime

# Initialize BigQuery client
client = bigquery.Client()

# Set your project ID and dataset name
PROJECT_ID = 'qwiklabs-gcp-04-af615cbd429c'
DATASET_ID = 'emergency_response_ml'
TABLE_ID = 'emergency_calls_response_times'
MODEL_ID = 'response_time_predictor'

print(f"Project ID: {PROJECT_ID}")
print(f"Dataset: {DATASET_ID}")

Project ID: qwiklabs-gcp-04-af615cbd429c
Dataset: emergency_response_ml


In [4]:
# Create dataset if it doesn't exist
dataset_id = f"{PROJECT_ID}.{DATASET_ID}"

try:
    client.get_dataset(dataset_id)
    print(f"Dataset {dataset_id} already exists.")
except:
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = "US"
    dataset = client.create_dataset(dataset, timeout=30)
    print(f"Created dataset {dataset_id}")

Created dataset qwiklabs-gcp-04-af615cbd429c.emergency_response_ml


In [6]:
# Load data from GCS into BigQuery table
table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

uri = "gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv"

load_job = client.load_table_from_uri(
    uri, table_id, job_config=job_config
)

load_job.result()

print(f"Loaded data from {uri} to {table_id}")

Loaded data from gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv to qwiklabs-gcp-04-af615cbd429c.emergency_response_ml.emergency_calls_response_times


In [7]:
# Preview the data
query = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
LIMIT 10
"""

df_preview = client.query(query).to_dataframe()
print("Data Preview:")
print(df_preview)
print("\n" + "="*80 + "\n")

Data Preview:
   call_id            call_timestamp call_type    location weather_condition  \
0    35957 2023-01-01 00:05:53+00:00      Fire    Highland             Rainy   
1    20832 2023-01-01 00:20:47+00:00      Fire     Oakmont             Rainy   
2    27949 2023-01-01 00:33:27+00:00      Fire   Riverside             Windy   
3    20199 2023-01-01 00:48:29+00:00      Fire   Riverside             Windy   
4    46938 2023-01-01 00:50:44+00:00    Rescue  Brookfield             Sunny   
5    17582 2023-01-01 02:28:50+00:00    Rescue    Downtown             Snowy   
6    21624 2023-01-01 02:44:06+00:00    Rescue     Oakmont             Snowy   
7    36793 2023-01-01 02:53:54+00:00      Fire   Riverside             Sunny   
8    41350 2023-01-01 03:52:33+00:00    Police  Greenfield             Windy   
9    32092 2023-01-01 04:09:23+00:00    Police   Maplewood             Snowy   

  day_of_week  time_of_day traffic_level  distance_to_station  \
0      Sunday            0          High

In [8]:
# Get schema information
table = client.get_table(table_id)
print("Schema:")
for field in table.schema:
    print(f"  {field.name}: {field.field_type}")
print("\n" + "="*80 + "\n")

Schema:
  call_id: INTEGER
  call_timestamp: TIMESTAMP
  call_type: STRING
  location: STRING
  weather_condition: STRING
  day_of_week: STRING
  time_of_day: INTEGER
  traffic_level: STRING
  distance_to_station: FLOAT
  units_available: INTEGER
  response_time: FLOAT




In [12]:
# Get basic statistics
query_stats = f"""
SELECT
    COUNT(*) as total_records,
    AVG(response_time) as avg_response_time,
    MIN(response_time) as min_response_time,
    MAX(response_time) as max_response_time,
    STDDEV(response_time) as stddev_response_time,
    COUNT(DISTINCT call_type) as num_call_types,
    COUNT(DISTINCT location) as num_locations,
    COUNT(DISTINCT weather_condition) as num_weather_conditions,
    AVG(distance_to_station) as avg_distance,
    AVG(units_available) as avg_units_available
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
"""

df_stats = client.query(query_stats).to_dataframe()
print("Data Statistics:")
print(df_stats.to_string())
print("\n" + "="*80 + "\n")

Data Statistics:
   total_records  avg_response_time  min_response_time  max_response_time  stddev_response_time  num_call_types  num_locations  num_weather_conditions  avg_distance  avg_units_available
0          50000          17.446134               2.01              36.55              5.296881               4              8                       4     15.079754               7.9979




In [13]:
# Check for null values
query_nulls = f"""
SELECT
    COUNTIF(response_time IS NULL) as null_response_time,
    COUNTIF(call_type IS NULL) as null_call_type,
    COUNTIF(location IS NULL) as null_location,
    COUNTIF(time_of_day IS NULL) as null_time_of_day,
    COUNTIF(day_of_week IS NULL) as null_day_of_week,
    COUNTIF(weather_condition IS NULL) as null_weather_condition,
    COUNTIF(distance_to_station IS NULL) as null_distance,
    COUNTIF(traffic_level IS NULL) as null_traffic_level,
    COUNTIF(units_available IS NULL) as null_units_available
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
"""

df_nulls = client.query(query_nulls).to_dataframe()
print("Null Value Check:")
print(df_nulls.to_string())
print("\n" + "="*80 + "\n")


Null Value Check:
   null_response_time  null_call_type  null_location  null_time_of_day  null_day_of_week  null_weather_condition  null_distance  null_traffic_level  null_units_available
0                   0               0              0                 0                 0                       0              0                   0                     0




In [14]:
# Show distribution of categorical variables
query_dist = f"""
SELECT
    call_type,
    COUNT(*) as count,
    AVG(response_time) as avg_response_time
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
GROUP BY call_type
ORDER BY count DESC
"""

df_dist = client.query(query_dist).to_dataframe()
print("Call Type Distribution:")
print(df_dist.to_string())

Call Type Distribution:
  call_type  count  avg_response_time
0      Fire  12585          17.434823
1    Police  12536          17.513849
2   Medical  12478          17.393902
3    Rescue  12401          17.441715


In [15]:
# Create a linear regression model to predict response times
model_id = f"{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}"

create_model_query = f"""
CREATE OR REPLACE MODEL `{model_id}`
OPTIONS(
    model_type='LINEAR_REG',
    input_label_cols=['response_time'],
    data_split_method='AUTO_SPLIT'
) AS
SELECT
    response_time,
    call_type,
    location,
    weather_condition,
    day_of_week,
    time_of_day,
    traffic_level,
    distance_to_station,
    units_available
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE response_time IS NOT NULL
"""

print("Creating model... This may take a few minutes.")
query_job = client.query(create_model_query)
query_job.result()  # Wait for completion

print(f"Model {model_id} created successfully!")

Creating model... This may take a few minutes.
Model qwiklabs-gcp-04-af615cbd429c.emergency_response_ml.response_time_predictor created successfully!


In [16]:
# Create synthetic test data and make predictions
predict_query = f"""
SELECT
    *
FROM
    ML.PREDICT(MODEL `{model_id}`,
        (
        SELECT
            'Medical Emergency' as call_type,
            'Downtown' as location,
            'Clear' as weather_condition,
            'Monday' as day_of_week,
            9 as time_of_day,  -- 9 AM
            'Medium' as traffic_level,
            5.2 as distance_to_station,
            3 as units_available
        UNION ALL
        SELECT
            'Fire' as call_type,
            'Suburbs' as location,
            'Rain' as weather_condition,
            'Saturday' as day_of_week,
            23 as time_of_day,  -- 11 PM
            'Low' as traffic_level,
            8.5 as distance_to_station,
            5 as units_available
        UNION ALL
        SELECT
            'Crime' as call_type,
            'Downtown' as location,
            'Clear' as weather_condition,
            'Friday' as day_of_week,
            18 as time_of_day,  -- 6 PM
            'High' as traffic_level,
            3.1 as distance_to_station,
            2 as units_available
        UNION ALL
        SELECT
            'Accident' as call_type,
            'Industrial' as location,
            'Fog' as weather_condition,
            'Wednesday' as day_of_week,
            14 as time_of_day,  -- 2 PM
            'Medium' as traffic_level,
            12.3 as distance_to_station,
            4 as units_available
        ))
"""

df_predictions = client.query(predict_query).to_dataframe()
print("Predictions on Synthetic Data:")
print("="*80)
print(df_predictions.to_string())
print("\n")

Predictions on Synthetic Data:
   predicted_response_time          call_type    location weather_condition day_of_week  time_of_day traffic_level  distance_to_station  units_available
0               366.602709  Medical Emergency    Downtown             Clear      Monday            9        Medium                  5.2                3
1             -4238.833333               Fire     Suburbs              Rain    Saturday           23           Low                  8.5                5
2               368.865737              Crime    Downtown             Clear      Friday           18          High                  3.1                2
3             -4234.903082           Accident  Industrial               Fog   Wednesday           14        Medium                 12.3                4




In [18]:
# Display in a more readable format
print("Prediction Summary:")
print("-" * 80)
for idx, row in df_predictions.iterrows():
    print(f"\nScenario {idx + 1}:")
    print(f"  Call Type: {row['call_type']}")
    print(f"  Location: {row['location']}")
    print(f"  Weather: {row['weather_condition']}")
    print(f"  Time: {row['time_of_day']}:00 on {row['day_of_week']}")
    print(f"  Traffic Level: {row['traffic_level']}")
    print(f"  Distance to Station: {row['distance_to_station']:.1f} km")
    print(f"  Units Available: {row['units_available']}")
    print(f"  Predicted Response Time: {row['predicted_response_time']:.2f} minutes")

Prediction Summary:
--------------------------------------------------------------------------------

Scenario 1:
  Call Type: Medical Emergency
  Location: Downtown
  Weather: Clear
  Time: 9:00 on Monday
  Traffic Level: Medium
  Distance to Station: 5.2 km
  Units Available: 3
  Predicted Response Time: 366.60 minutes

Scenario 2:
  Call Type: Fire
  Location: Suburbs
  Weather: Rain
  Time: 23:00 on Saturday
  Traffic Level: Low
  Distance to Station: 8.5 km
  Units Available: 5
  Predicted Response Time: -4238.83 minutes

Scenario 3:
  Call Type: Crime
  Location: Downtown
  Weather: Clear
  Time: 18:00 on Friday
  Traffic Level: High
  Distance to Station: 3.1 km
  Units Available: 2
  Predicted Response Time: 368.87 minutes

Scenario 4:
  Call Type: Accident
  Location: Industrial
  Weather: Fog
  Time: 14:00 on Wednesday
  Traffic Level: Medium
  Distance to Station: 12.3 km
  Units Available: 4
  Predicted Response Time: -4234.90 minutes
